In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv('AAPL.csv')


In [6]:
data = df['Close'].values.reshape(-1, 1)


KeyError: 'Close'

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)


In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length - 7): # Adjusted loop length for 7-day target
        seq = data[i:(i + seq_length), 0]
        target = data[(i + seq_length):(i + seq_length + 7), 0]
        if len(target) == 7:
            X.append(seq)
            y.append(target)
    return np.array(X), np.array(y)

SEQ_LENGTH = 60 # Using a 60-day lookback for better context
X, y = create_sequences(scaled_data, SEQ_LENGTH)

# Reshape X for the RNN input layer: [samples, time steps, features]
X = X.reshape(X.shape[0], X.shape[1], 1)

# --- 3. Split Data into Training and Testing Sets ---
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# --- 4. Build and Train the Simple RNN Model ---
model = Sequential([
    SimpleRNN(units=50, activation='relu', input_shape=(SEQ_LENGTH, 1)),
    Dense(units=7) # Output layer to predict the next 7 days
])

model.compile(optimizer='adam', loss='mse')

In [ ]:
print("Starting model training...")
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, verbose=1)
print("Model training complete.")

# --- 5. Make Predictions ---
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test)

# --- 6. Evaluate the Result ---
mae = mean_absolute_error(y_test_actual.flatten(), predictions.flatten())
rmse = np.sqrt(mean_squared_error(y_test_actual.flatten(), predictions.flatten()))


In [ ]:
print(f'\nModel Evaluation on Test Data:')
print(f'Mean Absolute Error (MAE): ${mae:.4f}')
print(f'Root Mean Squared Error (RMSE): ${rmse:.4f}')


In [ ]:
last_sequence = scaled_data[-SEQ_LENGTH:]
last_sequence = last_sequence.reshape(1, SEQ_LENGTH, 1)
future_prediction_scaled = model.predict(last_sequence)
future_prediction = scaler.inverse_transform(future_prediction_scaled)

print(f'\nPredicted AAPL closing values for the actual next 7 days:')
for i, val in enumerate(future_prediction[0]):
    print(f'Day {i+1}: ${val:.2f}')

In [ ]:
plt.figure(figsize=(10, 6))
# Plot actuals of the test set
# Take the first day of each prediction horizon to plot a continuous line of actuals
plt.plot(y_test_actual[:, 0], color='blue', label='Actual Test Prices (First Day of 7-day window)')
# Plot predictions of the test set
plt.plot(predictions[:, 0], color='red', linestyle='--', label='Predicted Test Prices (First Day of 7-day window)')
plt.title('AAPL Stock Price Prediction')
plt.xlabel('Time Steps in Test Set')
plt.ylabel('Stock Price ($)')
plt.legend()
plt.show()